# Clustering des images

Notebook extrait de `Recherche_textuelle-Clustering.ipynb`. Il conserve le setup et la g?n?ration d'embeddings n?cessaires, puis isole la partie clustering th?matique HDBSCAN/UMAP/refinement. Les sorties ont ?t? retir?es pour faciliter le versioning.

## 0) Setup & performance (GPU, batch, cache)


In [ ]:
import os, glob, time, math, json, hashlib, re
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

BATCH_IMG = 64 if DEVICE == "cuda" else 16
BATCH_TXT = 256 if DEVICE == "cuda" else 64
NUM_WORKERS = 0  # Windows: souvent 0 est plus stable
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)


## 1) Dataset / Images


In [ ]:
IMAGE_DIR = "data/images/val2017"
paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.*")))
print("Nombre d'images:", len(paths))
paths[:5]


## 2) Modèles : BLIP (caption) + OpenCLIP (embeddings)

En production :
- captions + embeddings sont calculés **en background** lors de l’upload (job queue).
- la recherche temps réel ne calcule que l’embedding du **prompt**.


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

BLIP_NAME = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(BLIP_NAME)
blip = BlipForConditionalGeneration.from_pretrained(BLIP_NAME).to(DEVICE).eval()


In [ ]:
import open_clip

CLIP_MODEL = "ViT-L-14"
CLIP_PRETRAIN = "openai"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAIN)
clip_model = clip_model.to(DEVICE).eval()
clip_tokenizer = open_clip.get_tokenizer(CLIP_MODEL)

with torch.no_grad():
    dummy = torch.randn(1,3,224,224).to(DEVICE)
    dim = clip_model.encode_image(dummy).shape[-1]
print("OpenCLIP dim:", dim)


## 3) Cache local (éviter de recalculer captions/embeddings)


In [ ]:
CACHE_DIR = "data/artifacts"
os.makedirs(CACHE_DIR, exist_ok=True)

CAPTIONS_PATH = os.path.join(CACHE_DIR, "captions_blip.json")
IMG_EMB_PATH  = os.path.join(CACHE_DIR, "emb_img.npy")
CAP_EMB_PATH  = os.path.join(CACHE_DIR, "emb_cap.npy")
META_PATH     = os.path.join(CACHE_DIR, "meta.json")

def file_signature(file_paths, max_files=200):
    sample = file_paths[:max_files]
    sig = []
    for p in sample:
        try:
            sig.append((os.path.basename(p), os.path.getsize(p)))
        except:
            sig.append((os.path.basename(p), None))
    return hashlib.md5(json.dumps(sig).encode()).hexdigest()

DATA_SIG = file_signature(paths)
DATA_SIG


## 4) Génération captions BLIP (batch, GPU si dispo)


In [ ]:
from PIL import Image
from tqdm import tqdm

def blip_caption_images(image_paths, batch_size=8):
    caps = []
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch = image_paths[i:i+batch_size]
        imgs = [Image.open(p).convert("RGB") for p in batch]
        inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = blip.generate(**inputs, max_new_tokens=30)
        batch_caps = processor.batch_decode(out, skip_special_tokens=True)
        caps.extend([c.strip() for c in batch_caps])
    return caps

captions = None
if os.path.exists(CAPTIONS_PATH) and os.path.exists(META_PATH):
    meta = json.load(open(META_PATH,"r",encoding="utf-8"))
    if meta.get("data_sig") == DATA_SIG:
        captions = json.load(open(CAPTIONS_PATH,"r",encoding="utf-8"))
        print(" captions loaded from cache")

if captions is None:
    captions = blip_caption_images(paths, batch_size=8 if DEVICE=="cuda" else 4)
    json.dump(captions, open(CAPTIONS_PATH,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump({"data_sig": DATA_SIG}, open(META_PATH,"w",encoding="utf-8"), indent=2)
    print("captions generated & cached")

captions[:5]


## 5) Embeddings OpenCLIP (batch + autocast GPU)


In [ ]:
def l2_normalize(x, eps=1e-12):
    import numpy as np
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)

@torch.no_grad()
def embed_images(image_paths, batch_size=BATCH_IMG):
    import numpy as np
    vecs = []
    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch = image_paths[i:i+batch_size]
        imgs = [clip_preprocess(Image.open(p).convert("RGB")) for p in batch]
        imgs = torch.stack(imgs).to(DEVICE)

        if DEVICE == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                v = clip_model.encode_image(imgs)
        else:
            v = clip_model.encode_image(imgs)

        vecs.append(v.float().cpu().numpy())
    return l2_normalize(np.vstack(vecs))

@torch.no_grad()
def embed_texts(texts, batch_size=BATCH_TXT):
    import numpy as np
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        tok = clip_tokenizer(batch).to(DEVICE)

        if DEVICE == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                v = clip_model.encode_text(tok)
        else:
            v = clip_model.encode_text(tok)

        vecs.append(v.float().cpu().numpy())
    return l2_normalize(np.vstack(vecs))

img_vecs = cap_vecs = None
if os.path.exists(IMG_EMB_PATH) and os.path.exists(CAP_EMB_PATH) and os.path.exists(META_PATH):
    meta = json.load(open(META_PATH,"r",encoding="utf-8"))
    if meta.get("data_sig") == DATA_SIG:
        img_vecs = np.load(IMG_EMB_PATH)
        cap_vecs = np.load(CAP_EMB_PATH)
        print(" embeddings loaded from cache")

if img_vecs is None or cap_vecs is None:
    img_vecs = embed_images(paths)
    cap_vecs = embed_texts(captions)
    np.save(IMG_EMB_PATH, img_vecs)
    np.save(CAP_EMB_PATH, cap_vecs)
    print(" embeddings generated & cached")

img_vecs.shape, cap_vecs.shape


# 8) Clustering thématique (HDBSCAN + UMAP 3D + refinement)


## amelioration avec DBCV

In [ ]:
import numpy as np
import math
import re
from collections import Counter

from hdbscan import HDBSCAN
import umap.umap_ as umap

import plotly.express as px
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import ollama
import json


In [ ]:
def l2_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def fuse_embeddings(img_vecs: np.ndarray, cap_vecs: np.ndarray, alpha: float = 0.7) -> np.ndarray:
    """
    Fused embedding = alpha*image + (1-alpha)*caption (both L2-normalized).
    Returns L2-normalized fused vectors.
    """
    img_n = l2_normalize(img_vecs.astype(np.float32))
    cap_n = l2_normalize(cap_vecs.astype(np.float32))
    fused = alpha * img_n + (1 - alpha) * cap_n
    return l2_normalize(fused)


In [ ]:
def cluster_stats(labels: np.ndarray):
    n = len(labels)
    noise = int(np.sum(labels == -1))
    noise_ratio = noise / n
    uniq = sorted(set(labels))
    n_clusters = len([c for c in uniq if c != -1])

    sizes = {c: int(np.sum(labels == c)) for c in uniq}
    top_sizes = sorted([(c, s) for c, s in sizes.items() if c != -1], key=lambda x: x[1], reverse=True)[:10]

    return {
        "N": n,
        "n_clusters": n_clusters,
        "noise_ratio": float(noise_ratio),
        "sizes": sizes,
        "top_cluster_sizes": top_sizes,
    }


In [ ]:
def get_dbcv(clusterer, X=None, labels=None):
    """
    Try to read DBCV from clusterer.relative_validity_ (common).
    Fallback: try validity_index if available.
    """
    # Most common / easiest
    if hasattr(clusterer, "relative_validity_"):
        try:
            val = float(clusterer.relative_validity_)
            if np.isfinite(val):
                return val
        except Exception:
            pass

    # Fallback if validity module exists in your hdbscan install
    try:
        from hdbscan.validity import validity_index
        if X is not None and labels is not None:
            return float(validity_index(X, labels, metric="euclidean"))
    except Exception:
        pass

    return float("nan")

def get_mean_cluster_stability(clusterer):
    """
    HDBSCAN produces a dict: clusterer.cluster_persistence_
    Higher = more stable clusters.
    """
    if hasattr(clusterer, "cluster_persistence_"):
        try:
            pers = np.array(clusterer.cluster_persistence_, dtype=np.float32)
            if pers.size > 0:
                return float(np.nanmean(pers))
        except Exception:
            pass
    return float("nan")


In [ ]:
def fit_hdbscan(X: np.ndarray, min_cluster_size: int, min_samples: int):
    """
    X must be L2-normalized. We use metric='euclidean' for performance & stability.
    """
    clusterer = HDBSCAN(
        min_cluster_size=int(min_cluster_size),
        min_samples=int(min_samples),
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )
    labels = clusterer.fit_predict(X)
    return labels, clusterer


In [ ]:
def auto_hdbscan_by_stability_dbcv(
    img_vecs: np.ndarray,
    cap_vecs: np.ndarray,
    alpha: float = 0.7,
    mcs_grid=None,
    ms_grid=None,
    target_noise=(0.10, 0.40),
    target_clusters=(5, 35),
    min_mean_stability=0.35,
    verbose=True
):
    """
    Returns: best_labels, best_model, report, X_fused
    """
    X = fuse_embeddings(img_vecs, cap_vecs, alpha=alpha)
    N = X.shape[0]

    if mcs_grid is None:
        base = int(np.clip(int(0.02 * N), 10, 80))
        mcs_grid = sorted(set([max(8, base//2), base, min(120, int(base*1.5)), min(180, base*2)]))
    if ms_grid is None:
        ms_grid = [3, 5, 8, 10]

    tried = []

    for mcs in mcs_grid:
        for ms in ms_grid:
            if ms > mcs:
                continue

            labels, model = fit_hdbscan(X, mcs, ms)
            st = cluster_stats(labels)

            mean_stab = get_mean_cluster_stability(model)
            dbcv = get_dbcv(model, X=X, labels=labels)

            tried.append({
                "mcs": int(mcs),
                "ms": int(ms),
                "labels": labels,
                "model": model,
                "stats": st,
                "mean_stability": mean_stab,
                "dbcv": dbcv
            })

    # 1) Filter by sanity constraints (noise + clusters)
    lo_noise, hi_noise = target_noise
    lo_c, hi_c = target_clusters

    filtered = []
    for t in tried:
        noise = t["stats"]["noise_ratio"]
        nc = t["stats"]["n_clusters"]
        if not (lo_noise <= noise <= hi_noise):
            continue
        if not (lo_c <= nc <= hi_c):
            continue
        # stability constraint
        if np.isfinite(t["mean_stability"]) and t["mean_stability"] >= min_mean_stability:
            filtered.append(t)

    # If filtered empty, relax stability constraint
    if len(filtered) == 0:
        filtered = sorted(tried, key=lambda x: (np.nan_to_num(x["mean_stability"], nan=-1.0)), reverse=True)[:10]

    # 2) Choose best by DBCV (fallback: stability)
    def sort_key(t):
        dbcv = np.nan_to_num(t["dbcv"], nan=-1e9)
        stab = np.nan_to_num(t["mean_stability"], nan=-1.0)
        # prefer higher DBCV, then stability
        return (dbcv, stab)

    filtered_sorted = sorted(filtered, key=sort_key, reverse=True)
    best = filtered_sorted[0]

    report = {
        "alpha": alpha,
        "selection": {
            "target_noise": target_noise,
            "target_clusters": target_clusters,
            "min_mean_stability": min_mean_stability
        },
        "best": {
            "min_cluster_size": best["mcs"],
            "min_samples": best["ms"],
            "dbcv": float(best["dbcv"]) if np.isfinite(best["dbcv"]) else None,
            "mean_stability": float(best["mean_stability"]) if np.isfinite(best["mean_stability"]) else None,
            **best["stats"]
        },
        "top_configs": [
            {
                "mcs": t["mcs"],
                "ms": t["ms"],
                "dbcv": float(t["dbcv"]) if np.isfinite(t["dbcv"]) else None,
                "mean_stability": float(t["mean_stability"]) if np.isfinite(t["mean_stability"]) else None,
                "n_clusters": t["stats"]["n_clusters"],
                "noise_ratio": t["stats"]["noise_ratio"],
                "top_cluster_sizes": t["stats"]["top_cluster_sizes"]
            }
            for t in filtered_sorted[:10]
        ]
    }

    if verbose:
        print(" Best config:", report["best"])
        print("Top configs:")
        for r in report["top_configs"][:5]:
            print(r)

    return best["labels"], best["model"], report, X


In [ ]:
def umap_3d(X: np.ndarray, n_neighbors=20, min_dist=0.05, metric="euclidean", random_state=42):
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric=metric,
        random_state=random_state
    )
    return reducer.fit_transform(X)

def plot_umap_3d(X3: np.ndarray, labels: np.ndarray, title="UMAP 3D • clusters"):
    df = {"x": X3[:,0], "y": X3[:,1], "z": X3[:,2], "cluster": labels.astype(int)}
    fig = px.scatter_3d(df, x="x", y="y", z="z", color="cluster", opacity=0.75, title=title)
    fig.show()


In [ ]:
def resize_cover(img: Image.Image, size=(320, 240)) -> Image.Image:
    return ImageOps.fit(img, size, method=Image.Resampling.LANCZOS, centering=(0.5,0.5))

def show_cluster_samples(cluster_id, labels, paths, captions=None, n=16, cols=4, tile=(320,240), title=None):
    idxs = np.where(labels == cluster_id)[0]
    if len(idxs) == 0:
        print("No images for this cluster.")
        return
    idxs = idxs[:n]

    rows = (len(idxs) + cols - 1) // cols
    plt.figure(figsize=(cols*4, rows*4))
    for i, idx in enumerate(idxs):
        img = Image.open(paths[idx]).convert("RGB")
        img = resize_cover(img, tile)
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"C{cluster_id} • #{i+1}", fontsize=10)

    if title:
        plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

    if captions is not None:
        print(f"Captions sample for cluster {cluster_id}:")
        for idx in idxs[:min(10, len(idxs))]:
            print("-", captions[idx][:140])


In [ ]:
def compute_cluster_centroids(X: np.ndarray, labels: np.ndarray):
    centroids = {}
    for cid in sorted(set(labels)):
        if cid == -1:
            continue
        idx = np.where(labels == cid)[0]
        if len(idx) == 0:
            continue
        cent = X[idx].mean(axis=0)
        cent = l2_normalize(cent.reshape(1,-1))[0]
        centroids[cid] = cent
    return centroids

def cluster_coherence_scores(X: np.ndarray, labels: np.ndarray):
    """
    Returns dict cluster_id -> {mean_sim, p10_sim, size}
    Similarity = dot(X, centroid) (X normalized)
    """
    centroids = compute_cluster_centroids(X, labels)
    out = {}
    for cid, cent in centroids.items():
        idx = np.where(labels == cid)[0]
        sims = X[idx] @ cent
        out[cid] = {
            "size": int(len(idx)),
            "mean_sim": float(np.mean(sims)),
            "p10_sim": float(np.quantile(sims, 0.10)),
        }
    return out


In [ ]:
CLUSTER_SYSTEM = """You are a photo-album clustering assistant for a professional photographer.
You only see text captions, not images.

Your job:
1) Infer a short cluster theme (3-6 words).
2) Provide 5-10 keywords.
3) Provide a short coherence check (high|medium|low) based ONLY on captions.
Return STRICT JSON only. Do NOT invent.
"""

def llm_label_cluster(cluster_id: int, captions_list: list[str], model="llama3.1:8b"):
    sample = captions_list[:25]
    text_block = "\n".join([f"- {c[:180]}" for c in sample])

    user_prompt = f"""
Cluster id: {cluster_id}

Captions sample:
{text_block}

Return JSON:
{{
  "cluster_id": {cluster_id},
  "theme": "short theme (3-6 words)",
  "keywords": ["k1","k2","k3"],
  "coherence": "high|medium|low",
  "notes": "short comment"
}}
"""

    resp = ollama.chat(
        model=model,
        messages=[{"role":"system","content":CLUSTER_SYSTEM},
                  {"role":"user","content":user_prompt}],
        options={"temperature":0.1}
    )

    txt = resp["message"]["content"].strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        s,e = txt.find("{"), txt.rfind("}")
        return json.loads(txt[s:e+1])


In [ ]:
def cosine_sim_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    return A @ B.T  # normalized

def refine_clusters_by_centroid(
    X: np.ndarray,
    labels: np.ndarray,
    outlier_quantile: float = 0.10,
    reassign_margin: float = 0.03
):
    labels_new = labels.copy()
    centroids = compute_cluster_centroids(X, labels_new)
    cluster_ids = sorted(centroids.keys())
    if not cluster_ids:
        return labels_new, {"moved":0, "to_noise":0}

    C = np.stack([centroids[c] for c in cluster_ids], axis=0)  # (K,d)
    moved = 0
    to_noise = 0

    for cid in cluster_ids:
        idx = np.where(labels_new == cid)[0]
        if len(idx) < 6:
            continue

        own_cent = centroids[cid]
        sims_own = X[idx] @ own_cent

        thr = np.quantile(sims_own, outlier_quantile)
        outliers = idx[sims_own <= thr]
        if len(outliers) == 0:
            continue

        sims_all = cosine_sim_matrix(X[outliers], C)
        best_k = np.argmax(sims_all, axis=1)
        best_sim = sims_all[np.arange(len(outliers)), best_k]
        best_cluster = [cluster_ids[k] for k in best_k]

        for i, pid in enumerate(outliers):
            current_sim = float(X[pid] @ own_cent)
            cand_c = best_cluster[i]
            cand_sim = float(best_sim[i])

            if cand_c != cid and (cand_sim - current_sim) >= reassign_margin:
                labels_new[pid] = cand_c
                moved += 1
            else:
                labels_new[pid] = -1
                to_noise += 1

    return labels_new, {"moved": moved, "to_noise": to_noise}


In [ ]:
# 1) Auto-selection via stability + DBCV
labels, model, report, X = auto_hdbscan_by_stability_dbcv(
    img_vecs, cap_vecs,
    alpha=0.7,
    target_noise=(0.10, 0.40),
    target_clusters=(5, 35),
    min_mean_stability=0.35,
    verbose=True
)

print("\nReport best:", report["best"])
print("Top clusters:", report["best"]["top_cluster_sizes"])

# 2) UMAP 3D interactive
X3 = umap_3d(X, n_neighbors=20, min_dist=0.05)
plot_umap_3d(X3, labels, title="UMAP 3D • HDBSCAN (selected by stability+DBCV)")

# 3) Refinement
labels_ref, moves = refine_clusters_by_centroid(X, labels, outlier_quantile=0.10, reassign_margin=0.03)
print("Refinement:", moves)

plot_umap_3d(X3, labels_ref, title="UMAP 3D • refined clusters")

# 4) Cluster semantic coherence scores (vector)
coh = cluster_coherence_scores(X, labels_ref)

# 5) LLM themes per cluster + full display
cluster_ids = [k for k,_ in Counter(labels_ref).most_common() if k != -1]

cluster_info = {}
for cid in cluster_ids:
    idx = np.where(labels_ref == cid)[0]
    caps = [captions[i] for i in idx]
    cluster_info[cid] = llm_label_cluster(cid, caps, model="llama3.1:8b")

# 6) Print + show samples
for cid in cluster_ids[:12]:
    info = cluster_info[cid]
    idx = np.where(labels_ref == cid)[0]
    print("="*80)
    print(f"Cluster {cid} | size={len(idx)} | mean_sim={coh[cid]['mean_sim']:.3f} | p10_sim={coh[cid]['p10_sim']:.3f}")
    print("Theme:", info.get("theme"))
    print("Keywords:", info.get("keywords", []))
    print("LLM coherence:", info.get("coherence"), "| Notes:", info.get("notes"))
    show_cluster_samples(
        cid, labels_ref, paths,
        captions=captions, n=20, cols=4,
        title=f"Cluster {cid} • {info.get('theme')}"
    )
